# 10 — SVM tuyến tính / RBF trên histogram màu (Luân)

Đặc trưng: histogram 16 bin/kênh (RGB hoặc HSV) + 6 thống kê/kênh (mean, std, skewness, energy,
smoothness, entropy — Gonzalez & Woods) = **66 chiều**, tính trên pixel trong mask Otsu (hoặc cả ảnh).

Bảng thí nghiệm: {RGB, HSV} × {linear, RBF} × {có, không mask} + KNN làm mốc, trên 2 cách chia:
- `mixed`: test lấy từ cả 3 dataset;
- `external`: train trên BananaID + GitHub, **kiểm định ngoài** trên BananaImageBD.

Tham số (C, gamma, k) chọn bằng GridSearchCV 5 fold **theo nhóm ảnh gần trùng trên tập train**, scoring macro-F1.
Std: bootstrap 1000 lần trên tập đánh giá (và std 5 fold CV).

**Kaggle:** Add Input = dataset output preprocessing. Bật Internet. Chạy hết → **Save Version**
(output `results/` sẽ được notebook 90 gộp với kết quả của Nguyên, Quỳnh).

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Kaggle: bật Settings -> Internet = On. Code lấy từ GitHub, dữ liệu lấy từ Kaggle Dataset (Add Input).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "luan"          # tên người chạy, ghi vào bảng kết quả chung

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "banana_ripeness").exists()), None)
if local is not None:                       # chạy trên máy, trong thư mục repo
    CODE_DIR = local
else:                                       # Kaggle / Colab: clone vào /tmp (không lẫn vào output notebook)
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
os.environ["BANANA_OWNER"] = OWNER
commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code:", CODE_DIR, "| commit", commit or "?")

%load_ext autoreload
%autoreload 2

In [ ]:
from banana_ripeness.data import find_data_root, load_manifest, get_split
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

# Tự tìm thư mục có manifest.csv trong /kaggle/input (Kaggle Dataset output preprocessing đã Add Input).
# Nếu có nhiều bản, chỉ rõ: DATA_ROOT = find_data_root("/kaggle/input/<ten-dataset>")
DATA_ROOT = find_data_root()
df = load_manifest(DATA_ROOT)
EXP = CODE_DIR / "configs" / "experiments.yaml"
print("Dữ liệu:", DATA_ROOT, "|", len(df), "ảnh")
pd.crosstab([df.split_mixed], df.label).loc[["train", "val", "test"]]

## 1. Đặc trưng (cho slide 5: nguồn, số bin, số chiều)

In [ ]:
from banana_ripeness.features import feature_summary, plot_color_features
display(feature_summary(bins=16))
for lb in ["Unripe", "Half_Ripe", "Ripe", "Overripe"]:
    r = df[df.label == lb].sample(1, random_state=42).iloc[0]
    plot_color_features(r.image_path, r.mask_path, bins=16, title=f"{r.dataset} | {lb}");

## 2. Chạy toàn bộ thí nghiệm

Danh sách nằm ở `configs/experiments.yaml` (mục `luan`). Phiên Kaggle bị ngắt thì chạy lại cell này: các lần đã xong sẽ được bỏ qua.

In [ ]:
from banana_ripeness.experiment import load_experiments, run_grid, load_results
specs = load_experiments(EXP, "luan")
res = run_grid(specs, df=df, data_root=DATA_ROOT, schemes=["mixed", "external"], owner=OWNER)
res = load_results()        # đọc lại từ /kaggle/working/results (có cột run_dir)
res = res[res.owner == OWNER]
len(res)

## 3. Bảng kết quả (mean ± std bootstrap)

In [ ]:
from banana_ripeness.experiment import compare_table, plot_comparison
REPORTS = Path("/kaggle/working/reports") if Path("/kaggle/working").exists() else Path("reports")
REPORTS.mkdir(parents=True, exist_ok=True)
print("=== scheme mixed — test (3 dataset) ===")
display(compare_table(res, split="test", scheme="mixed", save_path=REPORTS / "results_luan_mixed_test.csv"))
print("=== scheme external — kiểm định ngoài BananaImageBD ===")
display(compare_table(res, split="external", scheme="external", save_path=REPORTS / "results_luan_external.csv"))

In [ ]:
print("Std theo 5 fold CV trên train (tham số tốt nhất):")
display(compare_table(res, split="test", scheme="mixed", std="cv"))
plot_comparison(res, split="test", scheme="mixed");
plot_comparison(res, split="external", scheme="external");

## 4. Ảnh hưởng của không gian màu và tách nền

In [ ]:
from banana_ripeness.experiment import summarize_runs
s = summarize_runs(res)
for split, scheme in [("test", "mixed"), ("external", "external")]:
    t = s[(s.split == split) & (s.scheme == scheme)].copy()
    t["param_kernel"] = t["param_kernel"].fillna("-") if "param_kernel" in t else "-"
    print(f"=== {scheme} / {split}: macro-F1 ===")
    display(t.pivot_table(index=["model", "param_kernel"], columns=["param_space", "param_use_mask"],
                          values="f1_macro", dropna=False).round(3))

## 5. Mô hình tốt nhất (chọn theo **val**, không theo test)

In [ ]:
from banana_ripeness.experiment import best_run
from banana_ripeness.evaluation import top_confusions
import matplotlib.pyplot as plt
for scheme, split in [("mixed", "test"), ("external", "external")]:
    best = best_run(res, scheme=scheme, by_split="val")
    print(f"[{scheme}] tốt nhất theo val: {best.run_name} | tham số {best.record['fit_info']['best_params']}")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    best.plot_confusion("test", ax=axes[0]); best.plot_confusion(split, ax=axes[1]) if split != "test" else axes[1].axis("off")
    plt.tight_layout(); plt.show()
    display(top_confusions(best.metrics[split]["confusion_matrix"]))
    display(pd.DataFrame(best.metrics[split]["per_class"]).T.round(3))
    if "by_dataset" in best.metrics["test"]:
        print("macro-F1 test theo từng dataset:"); display(pd.DataFrame(best.metrics["test"]["by_dataset"]).T.round(3))

## 6. Ảnh demo: đoán đúng và nhầm Half_Ripe ↔ Ripe

In [ ]:
from banana_ripeness.experiment import prediction_grid
best = best_run(res, scheme="mixed", by_split="val")
p = best.predictions["test"]
prediction_grid(p, n=8, correct=True, data_root=DATA_ROOT);
prediction_grid(p, n=8, true="Half_Ripe", pred="Ripe", data_root=DATA_ROOT);
prediction_grid(p, n=8, true="Ripe", pred="Half_Ripe", data_root=DATA_ROOT);

## 7. Lưới tham số của mô hình tốt nhất (phụ lục)

In [ ]:
from banana_ripeness.models import load_model
m = load_model(best.run_dir)
print(m)
print("Kết quả CV (5 fold theo nhóm) đã lưu trong result.json:")
{k: v for k, v in best.record["fit_info"].items() if k.startswith("cv_")}